<a href="https://colab.research.google.com/github/Mary-S2/COM-PAPER-REVIEW/blob/main/loadandexplore.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import files
uploaded = files.upload()

Saving BBC News Train.csv.zip to BBC News Train.csv.zip


In [3]:
import pandas as pd

df = pd.read_csv('BBC News Train.csv.zip')
print(df.shape)
print(df.columns.tolist())
print(df['Category'].value_counts())

(1490, 3)
['ArticleId', 'Text', 'Category']
Category
sport            346
business         336
politics         274
entertainment    273
tech             261
Name: count, dtype: int64


In [4]:
print(df['Text'][0][:600])

worldcom ex-boss launches defence lawyers defending former worldcom chief bernie ebbers against a battery of fraud charges have called a company whistleblower as their first witness.  cynthia cooper  worldcom s ex-head of internal accounting  alerted directors to irregular accounting practices at the us telecoms giant in 2002. her warnings led to the collapse of the firm following the discovery of an $11bn (£5.7bn) accounting fraud. mr ebbers has pleaded not guilty to charges of fraud and conspiracy.  prosecution lawyers have argued that mr ebbers orchestrated a series of accounting tricks at 


In [5]:
df['word_count'] = df['Text'].str.split().str.len()
print(df['word_count'].describe())
print("Missing text:", df['Text'].isna().sum())
print("Duplicate texts:", df['Text'].duplicated().sum())

count    1490.000000
mean      385.012752
std       210.898616
min        90.000000
25%       253.000000
50%       337.000000
75%       468.750000
max      3345.000000
Name: word_count, dtype: float64
Missing text: 0
Duplicate texts: 50


In [6]:
import re

# 1. Do any duplicate texts have conflicting labels? (1 = labels agree)
dups = df[df['Text'].duplicated(keep=False)]
print("Max distinct labels per duplicated text:", dups.groupby('Text')['Category'].nunique().max())

# 2. Drop duplicates, keeping the first copy
df = df.drop_duplicates(subset='Text').reset_index(drop=True)
print("Shape after dedupe:", df.shape)

# 3. Collapse repeated spaces/newlines into single spaces
df['clean_text'] = df['Text'].apply(lambda t: re.sub(r'\s+', ' ', t).strip())
print(df['clean_text'][0][:300])

Max distinct labels per duplicated text: 1
Shape after dedupe: (1440, 4)
worldcom ex-boss launches defence lawyers defending former worldcom chief bernie ebbers against a battery of fraud charges have called a company whistleblower as their first witness. cynthia cooper worldcom s ex-head of internal accounting alerted directors to irregular accounting practices at the u


In [7]:
def chunk_text(text, chunk_size=150, overlap=30):
    words = text.split()
    step = chunk_size - overlap
    chunks = []
    for start in range(0, len(words), step):
        piece = words[start:start + chunk_size]
        chunks.append(" ".join(piece))
        if start + chunk_size >= len(words):
            break
    return chunks

rows = []
for _, r in df.iterrows():
    for i, c in enumerate(chunk_text(r['clean_text'])):
        rows.append({
            'chunk_id': f"{r['ArticleId']}_{i}",
            'ArticleId': r['ArticleId'],
            'Category': r['Category'],
            'chunk_text': c,
        })

chunks_df = pd.DataFrame(rows)
print(chunks_df.shape)
print(chunks_df['chunk_text'].str.split().str.len().describe())
print(chunks_df.iloc[0]['chunk_id'], '->', chunks_df.iloc[0]['chunk_text'][:200])

(4981, 4)
count    4981.000000
mean      132.692431
std        33.110327
min        31.000000
25%       136.000000
50%       150.000000
75%       150.000000
max       150.000000
Name: chunk_text, dtype: float64
1833_0 -> worldcom ex-boss launches defence lawyers defending former worldcom chief bernie ebbers against a battery of fraud charges have called a company whistleblower as their first witness. cynthia cooper wo


In [8]:
chunks_df.to_csv('bbc_chunks.csv', index=False)

from google.colab import files
files.download('bbc_chunks.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Build the index: one vector per chunk
vectorizer = TfidfVectorizer(stop_words='english', max_df=0.8, min_df=2)
tfidf_matrix = vectorizer.fit_transform(chunks_df['chunk_text'])
print("Index shape (chunks x vocabulary):", tfidf_matrix.shape)

def search_tfidf(query, k=5):
    q_vec = vectorizer.transform([query])
    scores = cosine_similarity(q_vec, tfidf_matrix).ravel()
    top = scores.argsort()[::-1][:k]
    results = chunks_df.iloc[top].copy()
    results['score'] = scores[top]
    return results[['chunk_id', 'Category', 'score', 'chunk_text']]

hits = search_tfidf("interest rates and inflation")
for _, h in hits.iterrows():
    print(f"[{h['chunk_id']}] ({h['Category']}) score={h['score']:.3f}")
    print(h['chunk_text'][:200], "...\n")

Index shape (chunks x vocabulary): (4981, 16759)
[1088_4] (business) score=0.411
report published on wednesday the bank said that with rates at their current level inflation would rise to its 2% target within two years. however boe governor mervyn king warned only last month that  ...

[1874_0] (business) score=0.376
bank voted 8-1 for no rate change the decision to keep interest rates on hold at 4.75% earlier this month was passed 8-1 by the bank of england s rate-setting body minutes have shown. one member of th ...

[1874_1] (business) score=0.359
in interest rates to 5%. he argued that economic growth was picking up and that the equity credit and housing markets had been stronger than expected. the bank s minutes said that risks to the inflati ...

[1088_3] (business) score=0.318
work to do in 2005 said avery shenfeld senior economist at cibc world markets. i think the federal reserve does not want to rock the boat and is using a gradual approach in raising the interest rate s ...


In [10]:
def show(query, k=3):
    print(f"\n=== {query} ===")
    for _, h in search_tfidf(query, k).iterrows():
        print(f"[{h['chunk_id']}] ({h['Category']}) score={h['score']:.3f}")
        print(h['chunk_text'][:150], "...")

show("cost of borrowing money")          # same idea as interest rates, different words
show("footballers signing new contracts")
show("computer viruses and online security")


=== cost of borrowing money ===
[1690_0] (politics) score=0.263
brown shrugs off economy fears gordon brown is to freeze petrol duty increases fund a £1bn package to avoid big council tax rises and boost childcare  ...
[192_1] (business) score=0.251
come. under direct threat will be the federal government s ability to make good on its debts to the social security trust fund. as soon as 2018 the fu ...
[334_0] (business) score=0.211
ecb holds rates amid growth fears the european central bank has left its key interest rate unchanged at 2% for the 19th month in succession. borrowing ...

=== footballers signing new contracts ===
[1127_2] (sport) score=0.153
senior appearances and scoring 25 goals before signing for sheffield united - where he scored 12 goals in 68 appearances. after signing for charlton h ...
[1831_4] (tech) score=0.151
and the first month under the new regime has seen many firms cautioned for not honouring them. some have been told to revamp websites so customers kno ..

In [11]:
!pip install -q sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(
    chunks_df['chunk_text'].tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)
print(embeddings.shape)

def search_embed(query, k=5):
    q = model.encode([query], normalize_embeddings=True)
    scores = (embeddings @ q.T).ravel()
    top = scores.argsort()[::-1][:k]
    results = chunks_df.iloc[top].copy()
    results['score'] = scores[top]
    return results[['chunk_id', 'Category', 'score', 'chunk_text']]

def show_embed(query, k=3):
    print(f"\n=== {query} ===")
    for _, h in search_embed(query, k).iterrows():
        print(f"[{h['chunk_id']}] ({h['Category']}) score={h['score']:.3f}")
        print(h['chunk_text'][:150], "...")

show_embed("cost of borrowing money")
show_embed("footballers signing new contracts")
show_embed("computer viruses and online security")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/78 [00:00<?, ?it/s]

In [ ]:
import numpy as np
np.save('bbc_embeddings.npy', embeddings)

from google.colab import files
files.download('bbc_embeddings.npy')

In [ ]:
!pip install -q faiss-cpu

In [ ]:
import faiss

index = faiss.IndexFlatIP(embeddings.shape[1])   # inner product = cosine for normalized vectors
index.add(embeddings.astype('float32'))
print("Vectors in index:", index.ntotal)

def search_faiss(query, k=5):
    q = model.encode([query], normalize_embeddings=True).astype('float32')
    scores, ids = index.search(q, k)
    results = chunks_df.iloc[ids[0]].copy()
    results['score'] = scores[0]
    return results[['chunk_id', 'Category', 'score', 'chunk_text']]

print(search_faiss("footballers signing new contracts", 3)[['chunk_id', 'Category', 'score']])

In [ ]:
!pip install -q transformers accelerate

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tok = AutoTokenizer.from_pretrained("google/flan-t5-base")
lm = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

def generate(prompt, max_new_tokens=100):
    inputs = tok(prompt, return_tensors="pt", truncation=True, max_length=1024)
    out = lm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tok.decode(out[0], skip_special_tokens=True)

In [ ]:
def answer_question(question, k=3):
    hits = search_faiss(question, k)

    chunks = []
    for _, h in hits.iterrows():
        article_id = h['chunk_id'].split('_')[0]
        chunks.append({
            "text": h['chunk_text'],
            "source": f"ArticleId {article_id} ({h['Category']}), chunk {h['chunk_id']}",
        })

    context = "\n\n".join(f"[{i+1}] {c['text']}" for i, c in enumerate(chunks))
    prompt = (
        "Answer the question using ONLY the numbered passages below. "
        "If the passages don't contain the answer, say you don't know.\n\n"
        f"Passages:\n{context}\n\n"
        f"Question: {question}\n"
        "Answer:"
    )

    answer = generate(prompt)

    print("Q:", question)
    print("A:", answer)
    print("\nGrounded in:")
    for i, c in enumerate(chunks):
        print(f"  [{i+1}] {c['source']}")

answer_question("What did Arsene Wenger agree to?")

In [ ]:
answer_question("What is Microsoft releasing to deal with viruses?")
answer_question("What happened to the Bank of England interest rate decision?")

In [ ]:
def read_chunk(cid, n=600):
    text = chunks_df.loc[chunks_df['chunk_id'] == cid, 'chunk_text'].iloc[0]
    print(f"--- {cid} ---\n{text[:n]}\n")

for cid in ['1057_1', '1057_2', '1588_0']:
    read_chunk(cid)

for cid in ['90_0', '90_1', '1874_0']:
    read_chunk(cid)

In [22]:
test_queries = {
    "business": ["Company fraud of millions of dollars", "Germeny's economic decline", "Positive or negative economies and the effect on families"],
    "sport": ["most unexpected sport come backs", "soccer players leaves his team", "recoveries from injuries from sports"],
    "tech": ["how to improve sales of gadgets", "motion control gaming controls", "internet speed in mb"],
    "politics": ["Corruption of uk keeping money from europe budget", "Iraq war when soldsmith wasnt advised", "noward truent snoker hall"],
    "entertainment": ["film director becomes officer", "meet the forckers film success in money", "channel audiences and what they watch more"],
}

def precision_at_k(search_fn, k=5):
    per_cat = {}
    for cat, queries in test_queries.items():
        scores = [(search_fn(q, k)['Category'] == cat).mean() for q in queries]
        per_cat[cat] = sum(scores) / len(scores)
    return per_cat, sum(per_cat.values()) / len(per_cat)

for name, fn in [("TF-IDF", search_tfidf), ("Embeddings (FAISS)", search_faiss)]:
    per_cat, overall = precision_at_k(fn)
    print(f"{name}: overall precision@5 = {overall:.3f}")
    for c, v in per_cat.items():
        print(f"   {c:14s} {v:.3f}")

TF-IDF: overall precision@5 = 0.627
   business       0.667
   sport          0.533
   tech           0.733
   politics       0.467
   entertainment  0.733
Embeddings (FAISS): overall precision@5 = 0.840
   business       0.800
   sport          1.000
   tech           0.933
   politics       0.667
   entertainment  0.800


In [23]:
for q in test_queries["politics"]:
    print(f"\n=== {q!r} ===")
    print("TF-IDF:")
    for _, h in search_tfidf(q, 5).iterrows():
        print(f"  [{h['chunk_id']}] ({h['Category']}) {h['chunk_text'][:80]}...")
    print("Embeddings:")
    for _, h in search_faiss(q, 5).iterrows():
        print(f"  [{h['chunk_id']}] ({h['Category']}) {h['chunk_text'][:80]}...")


=== 'Corruption of uk keeping money from europe budget' ===
TF-IDF:
  [1342_1] (business) was rampant during the shevardnadze era and boost georgia s fragile public finan...
  [1791_0] (politics) brown names 16 march for budget chancellor gordon brown will deliver his budget ...
  [1527_1] (politics) would be a political price to pay for the uk s stance - britain contributed more...
  [1534_2] (politics) to come to an end. mr straw said that as well as the veto over the rebate the uk...
  [192_0] (business) the ticking budget facing the us the budget proposals laid out by the administra...
Embeddings:
  [778_1] (politics) the court has now qualified its opinion on the community accounts for a decade i...
  [778_0] (politics) eu fraud clampdown urged eu member states are failing to report fraud and irregu...
  [778_2] (politics) eu presidency. any fraud in other member states is potentially fraud against the...
  [1534_0] (politics) uk rebate unjustified - chirac french president jacqu

In [24]:
!pip install -q pyspellchecker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 25.5 MB/s eta 0:00:00


In [25]:
from spellchecker import SpellChecker

spell = SpellChecker()

def clean_query(query):
    words = query.lower().split()
    corrected = []
    for w in words:
        fix = spell.correction(w)
        corrected.append(fix if fix else w)
    return " ".join(corrected)

# Test it on your messy ones
for q in ["noward truent snoker hall", "wenegr arsnal contrct", "soldsmith wasnt advised"]:
    print(f"{q!r} -> {clean_query(q)!r}")

'noward truent snoker hall' -> 'toward truest smoker hall'
'wenegr arsnal contrct' -> 'wiener arsenal contact'
'soldsmith wasnt advised' -> 'goldsmith want advised'


In [26]:
messy = "noward truent snoker hall"
fixed = clean_query(messy)
print("Fixed:", fixed)

print("\nTF-IDF before:", search_tfidf(messy, 5)['Category'].tolist())
print("TF-IDF after: ", search_tfidf(fixed, 5)['Category'].tolist())
print("\nEmbeddings before:", search_faiss(messy, 5)['Category'].tolist())
print("Embeddings after: ", search_faiss(fixed, 5)['Category'].tolist())

Fixed: toward truest smoker hall

TF-IDF before: ['politics', 'entertainment', 'tech', 'entertainment', 'entertainment']
TF-IDF after:  ['politics', 'entertainment', 'tech', 'entertainment', 'entertainment']

Embeddings before: ['sport', 'sport', 'sport', 'sport', 'sport']
Embeddings after:  ['entertainment', 'sport', 'entertainment', 'sport', 'sport']


In [27]:
from spellchecker import SpellChecker

# Build a dictionary from your own corpus, not generic English
corpus_words = " ".join(chunks_df['chunk_text'].tolist()).split()
spell = SpellChecker()
spell.word_frequency.load_words(corpus_words)  # teach it corpus-specific words

def clean_query_v2(query):
    words = query.lower().split()
    corrected = [spell.correction(w) or w for w in words]
    return " ".join(corrected)

fixed_v2 = clean_query_v2(messy)
print("Corrected (corpus-aware):", fixed_v2)

print("\nTF-IDF after v2:", search_tfidf(fixed_v2, 5)['chunk_id'].tolist())
print("Embeddings after v2:", search_faiss(fixed_v2, 5)['chunk_id'].tolist())

Corrected (corpus-aware): toward truest smoker hall

TF-IDF after v2: ['1582_0', '1167_0', '28_1', '2029_1', '714_2']
Embeddings after v2: ['714_6', '485_1', '714_5', '2173_4', '559_0']


In [28]:
for messy in ["noward truent snoker hall", "wenegr arsnal contrct", "soldsmith wasnt advised"]:
    fixed = clean_query_v2(messy)
    print(f"{messy!r} -> {fixed!r}")

'noward truent snoker hall' -> 'toward truest smoker hall'
'wenegr arsnal contrct' -> 'wenger arsenal contact'
'soldsmith wasnt advised' -> 'goldsmith want advised'


In [29]:
fixed_v2 = clean_query_v2("noward truent snoker hall")
print("Embeddings after v2:", search_faiss(fixed_v2, 5)['chunk_id'].tolist())

Embeddings after v2: ['714_6', '485_1', '714_5', '2173_4', '559_0']
